In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/02_api_football_client

In [0]:
import json
from pathlib import Path
from typing import Any


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide API settings, raw storage paths, secrets, countries, and quota
# limits are loaded from 00_project_config.
#
# HTTP communication, retries, pacing, rate-limit handling, and atomic JSON
# writing are provided by 02_api_football_client.

ENDPOINT = "venues"

PAGES_ROOT = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/venue_data/pages"
)

CONSOLIDATED_OUTPUT_FILE = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/venue_data/raw_venues.json"
)

MAX_REQUESTS_PER_RUN = (
    API_FOOTBALL_DAILY_REQUEST_LIMIT
    - API_FOOTBALL_DAILY_REQUEST_RESERVE
)


# -----------------------------------------------------------------------------
# Path helpers
# -----------------------------------------------------------------------------

def normalize_country_for_path(
    country: str,
) -> str:
    """
    Convert a country name into a filesystem-friendly identifier.

    Example:
        "United Kingdom" -> "united_kingdom"
    """

    return (
        country
        .strip()
        .lower()
        .replace(" ", "_")
    )


def get_page_file_path(
    country: str,
    page: int,
) -> Path:
    """
    Return the raw path for one country/page API response.

    Each successful page becomes a durable checkpoint so reruns only request
    pages that are still missing.
    """

    country_path = normalize_country_for_path(
        country
    )

    return Path(
        PAGES_ROOT,
        f"country={country_path}",
        f"page={page:04d}.json",
    )


# -----------------------------------------------------------------------------
# JSON helpers
# -----------------------------------------------------------------------------

def read_json_file(
    file_path: Path,
) -> dict[str, Any]:
    """Read and validate one persisted raw JSON response."""

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError(
            f"Expected JSON object in raw file: {file_path}"
        )

    return payload


# -----------------------------------------------------------------------------
# API response validation
# -----------------------------------------------------------------------------

def validate_venue_page(
    payload: dict[str, Any],
    *,
    country: str,
    expected_page: int,
) -> tuple[list[dict[str, Any]], int, int]:
    """
    Validate one API-Football venues response.

    Returns:
        venue records
        current page
        total pages
    """

    venues = payload.get("response")
    paging = payload.get("paging")

    if not isinstance(venues, list):
        raise APIFootballError(
            "Expected API-Football 'response' to be a list for "
            f"country={country}, page={expected_page}."
        )

    if not isinstance(paging, dict):
        raise APIFootballError(
            "Expected API-Football 'paging' to be an object for "
            f"country={country}, page={expected_page}."
        )

    try:
        current_page = int(
            paging.get("current")
        )

        total_pages = int(
            paging.get("total")
        )

    except (TypeError, ValueError) as exc:
        raise APIFootballError(
            "Invalid pagination metadata for "
            f"country={country}, page={expected_page}."
        ) from exc

    if current_page != expected_page:
        raise APIFootballError(
            "API-Football returned an unexpected page number. "
            f"Requested={expected_page}, "
            f"returned={current_page}, "
            f"country={country}."
        )

    if total_pages < 1:
        raise APIFootballError(
            "API-Football returned an invalid total page count "
            f"for country={country}: {total_pages}"
        )

    if current_page > total_pages:
        raise APIFootballError(
            "API-Football returned current page greater than total pages "
            f"for country={country}: "
            f"{current_page}/{total_pages}"
        )

    # For the configured football countries we expect venue reference data.
    # Avoid treating an unexpected empty response as a valid completed page.
    if not venues:
        raise APIFootballError(
            "API-Football returned zero venues for "
            f"country={country}, page={expected_page}."
        )

    return (
        venues,
        current_page,
        total_pages,
    )


# -----------------------------------------------------------------------------
# Completion helpers
# -----------------------------------------------------------------------------

def get_saved_total_pages(
    country: str,
) -> int | None:
    """
    Return the expected page count from the saved first page.

    If page 1 does not exist, the country has not yet started.
    """

    first_page_file = get_page_file_path(
        country,
        1,
    )

    if not first_page_file.exists():
        return None

    payload = read_json_file(
        first_page_file
    )

    _, _, total_pages = validate_venue_page(
        payload,
        country=country,
        expected_page=1,
    )

    return total_pages


def country_is_complete(
    country: str,
) -> bool:
    """
    Return True when every expected venue page for a country exists locally.
    """

    total_pages = get_saved_total_pages(
        country
    )

    if total_pages is None:
        return False

    return all(
        get_page_file_path(
            country,
            page,
        ).exists()
        for page in range(
            1,
            total_pages + 1,
        )
    )


def all_extractions_complete() -> bool:
    """
    Return True when every configured country has been fully downloaded.
    """

    return all(
        country_is_complete(
            country
        )
        for country in API_FOOTBALL_VENUE_COUNTRIES
    )


# -----------------------------------------------------------------------------
# Consolidation helper
# -----------------------------------------------------------------------------

def build_consolidated_venue_file() -> int:
    """
    Build the existing consolidated raw venue file from persisted page files.

    No API requests are made here.

    Full API envelopes remain stored in the page-level raw area for lineage.
    The consolidated output contains only venue business records because this
    matches the structure currently consumed by Bronze.

    Duplicate venue IDs are removed defensively during consolidation.

    Returns:
        Number of unique venues written.
    """

    venues_by_id: dict[int, dict[str, Any]] = {}

    for country in API_FOOTBALL_VENUE_COUNTRIES:

        total_pages = get_saved_total_pages(
            country
        )

        if total_pages is None:
            raise RuntimeError(
                "Cannot build consolidated venue file because "
                f"country={country} has no saved first page."
            )

        for page in range(
            1,
            total_pages + 1,
        ):

            page_file = get_page_file_path(
                country,
                page,
            )

            if not page_file.exists():
                raise RuntimeError(
                    "Cannot build consolidated venue file because "
                    f"page is missing: {page_file}"
                )

            payload = read_json_file(
                page_file
            )

            venues, _, _ = validate_venue_page(
                payload,
                country=country,
                expected_page=page,
            )

            for venue in venues:

                venue_id = venue.get("id")

                if venue_id is None:
                    raise RuntimeError(
                        "Venue record is missing its source ID for "
                        f"country={country}, page={page}."
                    )

                try:
                    venue_id = int(
                        venue_id
                    )

                except (TypeError, ValueError) as exc:
                    raise RuntimeError(
                        "Venue record contains an invalid source ID for "
                        f"country={country}, page={page}: {venue_id}"
                    ) from exc

                venues_by_id[venue_id] = venue

    if not venues_by_id:
        raise RuntimeError(
            "Consolidated venue dataset would be empty."
        )

    # Sort by source ID to make the generated file deterministic and easier
    # to review in Git/debugging tools.
    consolidated_venues = [
        venues_by_id[venue_id]
        for venue_id in sorted(
            venues_by_id
        )
    ]

    write_json_atomic(
        data=consolidated_venues,
        file_path=CONSOLIDATED_OUTPUT_FILE,
    )

    return len(
        consolidated_venues
    )


# -----------------------------------------------------------------------------
# Load API credential
# -----------------------------------------------------------------------------
# Retrieve the API key from Databricks Secrets rather than storing credentials
# directly in notebook source code.

api_key = dbutils.secrets.get(
    scope=API_FOOTBALL_SECRET_SCOPE,
    key=API_FOOTBALL_SECRET_KEY,
)


# -----------------------------------------------------------------------------
# Ingestion state
# -----------------------------------------------------------------------------

requests_made_this_run = 0
pages_downloaded_this_run = 0
pages_reused = 0

stop_for_daily_budget = False


# -----------------------------------------------------------------------------
# Fetch venues
# -----------------------------------------------------------------------------
# Processing order is deterministic:
#
#   country -> page
#
# Existing valid page files are treated as checkpoints and do not generate
# additional API requests.

with APIFootballClient(
    api_key=api_key,
    base_url=API_FOOTBALL_BASE_URL,
    min_request_interval_seconds=API_FOOTBALL_MIN_REQUEST_INTERVAL_SECONDS,
) as client:

    for country in API_FOOTBALL_VENUE_COUNTRIES:

        if stop_for_daily_budget:
            break

        print(
            "\n"
            "------------------------------------------------------------"
        )

        print(
            f"Processing venues for {country}"
        )


        # ---------------------------------------------------------------------
        # Skip fully completed countries
        # ---------------------------------------------------------------------

        if country_is_complete(
            country
        ):

            total_pages = get_saved_total_pages(
                country
            )

            print(
                f"Country already complete: {total_pages} page(s). "
                "No API requests required."
            )

            continue


        # ---------------------------------------------------------------------
        # Process pages
        # ---------------------------------------------------------------------

        page = 1

        while True:

            page_file = get_page_file_path(
                country,
                page,
            )


            # -----------------------------------------------------------------
            # Reuse an existing raw page
            # -----------------------------------------------------------------

            if page_file.exists():

                payload = read_json_file(
                    page_file
                )

                (
                    venues,
                    current_page,
                    total_pages,
                ) = validate_venue_page(
                    payload,
                    country=country,
                    expected_page=page,
                )

                pages_reused += 1

                print(
                    f"Page {current_page}/{total_pages} already exists "
                    f"({len(venues)} venues). Skipping request."
                )


            else:

                # -------------------------------------------------------------
                # Protect configured request budget
                # -------------------------------------------------------------

                if requests_made_this_run >= MAX_REQUESTS_PER_RUN:

                    print(
                        "\nConfigured daily request budget reached."
                    )

                    stop_for_daily_budget = True
                    break


                # -------------------------------------------------------------
                # Protect actual remaining API quota
                # -------------------------------------------------------------
                # The response headers are authoritative because another
                # notebook may already have consumed today's allowance.

                if (
                    client.daily_remaining is not None
                    and client.daily_remaining
                    <= API_FOOTBALL_DAILY_REQUEST_RESERVE
                ):

                    print(
                        "\nAPI-Football daily quota reached the "
                        "configured safety reserve."
                    )

                    stop_for_daily_budget = True
                    break


                # -------------------------------------------------------------
                # Fetch one venue page
                # -------------------------------------------------------------

                print(
                    f"Fetching page {page}..."
                )

                payload = client.get(
                    endpoint=ENDPOINT,
                    params={
                        "country": country,
                        "page": page,
                    },
                )

                requests_made_this_run += 1


                # -------------------------------------------------------------
                # Validate before persisting
                # -------------------------------------------------------------

                (
                    venues,
                    current_page,
                    total_pages,
                ) = validate_venue_page(
                    payload,
                    country=country,
                    expected_page=page,
                )


                # -------------------------------------------------------------
                # Persist the complete API envelope
                # -------------------------------------------------------------
                # Keeping the complete response preserves:
                #
                #   get
                #   parameters
                #   errors
                #   results
                #   paging
                #   response

                write_json_atomic(
                    data=payload,
                    file_path=str(page_file),
                )

                pages_downloaded_this_run += 1

                print(
                    f"Saved page {current_page}/{total_pages}: "
                    f"{len(venues)} venues."
                )

                print(
                    "API daily requests remaining: "
                    f"{client.daily_remaining}"
                )


            # -----------------------------------------------------------------
            # Finish this country or continue pagination
            # -----------------------------------------------------------------

            if current_page >= total_pages:

                print(
                    f"Completed {country}: "
                    f"{total_pages} page(s)."
                )

                break

            page = current_page + 1


# -----------------------------------------------------------------------------
# Run summary
# -----------------------------------------------------------------------------

completed_countries = sum(
    1
    for country in API_FOOTBALL_VENUE_COUNTRIES
    if country_is_complete(
        country
    )
)

print(
    "\n"
    "============================================================"
)

print(
    "Venue ingestion summary"
)

print(
    "============================================================"
)

print(
    f"Requests made this run: {requests_made_this_run}"
)

print(
    f"New pages downloaded: {pages_downloaded_this_run}"
)

print(
    f"Existing pages reused: {pages_reused}"
)

print(
    f"Completed countries: "
    f"{completed_countries}/"
    f"{len(API_FOOTBALL_VENUE_COUNTRIES)}"
)


# -----------------------------------------------------------------------------
# Build consolidated raw dataset
# -----------------------------------------------------------------------------
# Only replace raw_venues.json after every configured country is complete.
#
# This prevents Bronze from accidentally consuming a partially refreshed
# reference dataset.

if all_extractions_complete():

    venue_count = build_consolidated_venue_file()

    print(
        "\nAll configured venue extractions are complete."
    )

    print(
        f"Consolidated {venue_count} unique venues."
    )

    print(
        f"Consolidated raw file: {CONSOLIDATED_OUTPUT_FILE}"
    )

else:

    print(
        "\nVenue extraction is not complete yet."
    )

    print(
        "Run this notebook again when additional API quota is available."
    )

    print(
        "Previously downloaded pages will be reused automatically."
    )